## Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os 

from dotenv import load_dotenv

load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

response = model.invoke("Why do parrots talk?")

response.content

'Parrots talk for a fascinating combination of reasons, driven by their intelligence, social nature, and the way their brains are wired. It\'s not just about mimicry; there\'s a lot more going on. Here\'s a breakdown of the key factors:\n\n**1. Social Bonding and Communication:**\n\n* **Flock Animals:** Parrots are highly social creatures that live in large flocks in the wild. Communication is vital for survival, coordinating activities like foraging, warning of predators, and maintaining group cohesion.\n* **Mimicry as a Social Tool:** In their natural environment, parrots often mimic the calls of other birds in their flock. This helps them blend in, identify members, and build stronger bonds. They essentially learn the "language" of their group.\n* **Strengthening Relationships:** When a parrot talks to its human "flock," it\'s often an extension of this social instinct. They are trying to communicate with you, get your attention, and strengthen their bond with you.\n\n**2. Intellige

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):

    title: str=Field(description="The title of the movie")

    yeaer: int=Field(description="This year movie was released")

    director: str=Field(description="The director of the movie")

    rating: float=Field(description="The movies ratings of out 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x7f3196d0a750>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'st

In [5]:
## Without strucutured output 
model.invoke("Gimme details of movie Salaar")

AIMessage(content='Okay, let\'s dive into the details of the highly anticipated Indian action film **Salaar: Part 1 – Ceasefire**.\n\n**Key Information:**\n\n*   **Title:** Salaar: Part 1 – Ceasefire\n*   **Language:** Telugu (with dubbed versions in Hindi, Tamil, Malayalam, and Kannada)\n*   **Genre:** Action, Thriller\n*   **Release Date:** December 22, 2023\n*   **Director:** Prashanth Neel\n*   **Producers:** Vijay Kiragandur under Hombale Films\n*   **Main Cast:**\n    *   Prabhas as Deva / Salaar\n    *   Prithviraj Sukumaran as Vardharaja Mannar\n    *   Shruti Haasan as Aadhya\n    *   Tinnu Anand as Khaleel\n    *   Jagapathi Babu as Raja Mannar\n    *   Easwari Rao as Mother of Salaar\n    *   Sriya Reddy as Radhika\n    *   Ramachandra Raju as Faizal\n    *   Mime Gopi as Dorababu\n    *   Prudhvi Raj as Rajanna\n\n**Synopsis/Plot (without major spoilers):**\n\n"Salaar: Part 1 – Ceasefire" is set in the fictional, war-torn, and dystopian city of **Khansaar**. The story revol

In [6]:
model_with_structure.invoke("Gimme details of movie Salaar")

Movie(title='Salaar', yeaer=2023, director='Prashanth Neel', rating=7.7)

### Message Output alongside parsed structure


In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title: str=Field(..., description="The title of the movie")

    yeaer: int=Field(...,description="This year movie was released")

    director: str=Field(...,description="The director of the movie")

    rating: float=Field(...,description="The movies ratings of out 10")


model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Gimme details of movie Salaar")

response

{'raw': AIMessage(content='{\n  "title": "Salaar",\n  "yeaer": 2023,\n  "director": "Prashanth Neel",\n  "rating": 7.5\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e6914-f785-79b2-a133-7452d30b6df4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 44, 'total_tokens': 52, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Salaar', yeaer=2023, director='Prashanth Neel', rating=7.5),
 'parsing_error': None}

### Nested Struture

In [8]:
from pydantic import BaseModel, Field


class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Gimme details of movie Salaar")

response

MovieDetails(title='Salaar', year=2023, cast=[Actor(name='Prabhas', role='Devaratha Raisa'), Actor(name='Shruti Haasan', role='Aadhya'), Actor(name='Prithviraj Sukumaran', role='Varadharaja Mannar'), Actor(name='Jagapathi Babu', role='Raja Mannar')], genres=['Action', 'Crime', 'Drama'], budget=200.0)

In [9]:
for n in response.cast:
    print(n.name)  # as it's a object we use . nottation

Prabhas
Shruti Haasan
Prithviraj Sukumaran
Jagapathi Babu


### TypeDict

TypeDict provides a simpler alternatives usin gPython's built in typing, ideal when you dont need runtime validations


In [10]:
from typing_extensions import TypedDict, Annotated


class MovieDic(TypedDict):

    title: Annotated[str, ..., "The title of the movie"]

    year: Annotated[int, ..., "This year movie was released"]

    director: Annotated[str, ..., "The director of the movie"]

    rating: Annotated[float, ..., "The movies ratings of out 10"]

In [11]:
model_with_type_dict = model.with_structured_output(MovieDic)

response = model_with_type_dict.invoke("Gimme details of the movie Iron Man 1")

response

{'title': 'Iron Man', 'year': 2008, 'director': 'Jon Favreau', 'rating': 7.9}

In [12]:

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Gimme details of movie Iron Man 3")

response

{'title': 'Iron Man 3',
 'year': 2013,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Gwyneth Paltrow', 'role': 'Pepper Potts'},
  {'name': 'Don Cheadle', 'role': 'James "Rhodey" Rhodes / Iron Patriot'},
  {'name': 'Guy Pearce', 'role': 'Aldrich Killian'},
  {'name': 'Rebecca Hall', 'role': 'Maya Hansen'}],
 'genres': ['Action', 'Sci-Fi', 'Adventure'],
 'budget': 200000000.0}

In [13]:
for celebrity in response["cast"]:
    print(celebrity['name'])  ## as it's a dict we use ['']


Robert Downey Jr.
Gwyneth Paltrow
Don Cheadle
Guy Pearce
Rebecca Hall


In [14]:
model.profile

{'name': 'Gemini 2.5 Flash-Lite',
 'release_date': '2025-06-17',
 'last_updated': '2025-06-17',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

In [15]:
model_with_type_dict.profile

AttributeError: 'RunnableSequence' object has no attribute 'profile'

### Data classes

A data class is a class typically containing maily data, although there aren't really any restrictions, You create it using the @dataclass decorator

In [17]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""

    name: str=Field(description="The name of a person")
    email: str=Field(description="The email address of a person")
    phone: str=Field(description="The phone number of a person")


agent= create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: John doe, jhon@example.com, (555) 123-4567 "}]
})



result


{'messages': [HumanMessage(content='Extract contact info from: John doe, jhon@example.com, (555) 123-4567 ', additional_kwargs={}, response_metadata={}, id='1d327cad-dbf8-49b5-a7b8-8aa094046230'),
  AIMessage(content='{\n  "name": "John doe",\n  "email": "jhon@example.com",\n  "phone": "(555) 123-4567"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e6916-370b-7722-894d-d2eacad176f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 45, 'total_tokens': 76, 'input_token_details': {'cache_read': 0}})],
 'structured_response': ContactInfo(name='John doe', email='jhon@example.com', phone='(555) 123-4567')}

In [19]:
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""

    name: str
    email: str
    phone: str

agent= create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: John doe, jhon@example.com, (555) 123-4567 "}]
})


result['structured_response']


{'name': 'John doe', 'email': 'jhon@example.com', 'phone': '(555) 123-4567'}

In [20]:
## Dataclass


from dataclasses import dataclass

from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""

    name: str
    email: str
    phone: str



agent= create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: John doe, jhon@example.com, (555) 123-4567 "}]
})


result['structured_response']


ContactInfo(name='John doe', email='jhon@example.com', phone='(555) 123-4567')